# Beer Game: token constraints, information sharing and bullwhip

**Start here. This notebook implements the new `ThoughtsReflection_Paragraph.docx`, not the earlier training experiment.**

1. **H1:** short preparation budgets reduce the fitted weight on outstanding replenishment (beta).
2. **H2:** short preparation budgets increase team holding/backlog cost.
3. Both conditions share all four roles' current inventories. There is **no trained/control condition**.
4. The same GPT-5.4 nano snapshot uses `reasoning.effort="none"` in both conditions. The API enforces different **visible memo-token caps**, followed by an identical **192-token structured final answer**. This is an operational proxy for constrained preparation, not a measure of human seconds or total internal cognition.
5. **Default: free offline calibration.** No API key or payment is needed. API calibration comes before a small game pilot; no automatic large study.

The source document is [here](../data/source/ThoughtsReflection_Paragraph.docx). Full protocol: [docs/PROTOCOL.md](../docs/PROTOCOL.md).

## 1. Setup (once)

Use a Python 3.9+ Jupyter kernel. If imports fail, run the commented installation line below, then restart the kernel. Never paste an API key into notebook source or outputs.

The setup and report cells install missing Word/Excel export packages into the active kernel environment. A report-export error does not require rerunning the simulation.

In [1]:
from pathlib import Path
import sys, os, getpass, importlib
from IPython.display import display, Markdown, Image
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for parent in candidates for p in
    (parent, parent / "beer_game_experiment", parent / "Project" / "beer_game_experiment")
    if (p / "src" / "beergame_study").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the beer_game_experiment project.")
# If needed, uncomment this line, run it, then restart the kernel:
# %pip install -r {ROOT / "requirements.txt"}
sys.path.insert(0, str(ROOT / "src"))
from beergame_study import Config, run, run_directory, report, software_checks
from beergame_study.api import StudyStopped
print("Project:", ROOT)

# Report exporters must be installed in THIS notebook's Python environment.
def ensure_report_dependencies():
    import importlib.util, subprocess
    packages = {"docx": "python-docx>=1,<2", "openpyxl": "openpyxl>=3,<4"}
    missing = [package for module, package in packages.items()
               if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing missing report packages into:", sys.executable)
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
        importlib.invalidate_caches()
    return True

ensure_report_dependencies()


Project: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment


True

## 2. Settings and cost

- `MODE="offline"`: deterministic software check, **not GPT evidence**. The same synthetic rule operates in both conditions.
- `MODE="api"`, `STAGE="calibration"`: 24 independent hypothetical states × reference/short/adequate/doubled allowance × two stages = **192 planned calls**. P25/P95 of reference memo use suggest candidate caps; the screen checks truncation and benchmark differences. This economical screen differs from the reflection's four-chain human calibration.
- After a **passed API calibration**, set `STAGE="game"`, copy its suggested caps and folder into the settings. Start with `GROUPS_PER_CONDITION=1` (two chains, eight agents). Use **2 for a four-chain, 16-agent pilot**. The reflection's main design uses **10 per condition = 20 chains = 80 agents**; do not automatically scale.
- Game: 40 weeks, first four fixed-order warmup weeks, then 36 decision weeks. Another 12 weeks are scored automatically, with no API calls.
- Rates checked 2026-10-04: $0.20 input / $0.02 cached input / $1.25 output per million tokens. [Official model documentation](https://developers.openai.com/api/docs/models/gpt-5.4-nano). Cost estimates below assume 1,500 input tokens/call; actual cost varies with history and output. The local allowance is a guard, not your account balance.

**Resume:** keep experiment settings and replicate ID fixed; raise only the dollar allowance if required. Scientific/code changes create a new folder. No earlier experiment data are reused.

In [6]:
MODE = "api"                # Change to "api" only when ready to spend
STAGE = "calibration"           # "calibration" first, then "game"
GROUPS_PER_CONDITION = 1         # 1=8 agents, 2=16 agents, 10=80 agents (two conditions)
SHORT_TOKENS = 64               # Provisional; replace with API calibration result
ADEQUATE_TOKENS = 256           # Provisional; replace with API calibration result
CALIBRATION_RUN = ""            # Paste passed API calibration folder for STAGE="game"
COLLECT_DEBRIEFS = False         # Optional post-game questionnaire; +1 call per agent
BUDGET_USD = 1.05               # Total allowance for THIS run, including cached spending
REPLICATE_ID = "token-pilot-01"
CONFIG = Config(mode=MODE, stage=STAGE, replicate_id=REPLICATE_ID,
    groups_per_condition=GROUPS_PER_CONDITION, short_tokens=SHORT_TOKENS,
    adequate_tokens=ADEQUATE_TOKENS, calibration_run=CALIBRATION_RUN,
    budget_usd=BUDGET_USD, collect_debriefs=COLLECT_DEBRIEFS)
CONFIG.validate()
run_dir = run_directory(ROOT, CONFIG)
planned = 192 if STAGE == "calibration" else 8 * GROUPS_PER_CONDITION * (1 + 2 * 36 + int(COLLECT_DEBRIEFS))
output_allowance = (24 * (1024 + SHORT_TOKENS + ADEQUATE_TOKENS + 2*ADEQUATE_TOKENS + 4*192)
                    if STAGE == "calibration" else
                    GROUPS_PER_CONDITION * (4*36*(SHORT_TOKENS+ADEQUATE_TOKENS+2*192) + 8*192 + 8*384*int(COLLECT_DEBRIEFS)))
scenario = (planned*1500*.20 + output_allowance*1.25)/1e6
print(f"{MODE} / {STAGE}; {planned} planned calls plus transport retries/replacements")
print(f"Illustrative cost at 1,500 input tokens/call and full output use: ${scenario:.2f}; not a quote.")
print(f"Request pacing alone: at least {planned * CONFIG.request_interval_s / 60:.1f} minutes for an uncached run.")
print("Local allowance:", BUDGET_USD, "USD; result folder:", run_dir)

api / calibration; 192 planned calls plus transport retries/replacements
Illustrative cost at 1,500 input tokens/call and full output use: $0.14; not a quote.
Request pacing alone: at least 3.2 minutes for an uncached run.
Local allowance: 1.05 USD; result folder: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6778221cada613b6


## 3. Free checks before any paid call

Checks include simulator conservation, final-order consequences, beta recovery with legitimate zero orders, and a sensible benchmark outperforming always-zero/always-large ordering. They verify the software, not GPT behavior.

In [7]:
checks = software_checks()
display(checks)
assert checks.passed.all()
print("Free checks passed. No API calls made by this cell.")

,check,passed,detail
0,Simple demand benchmark beats always-zero and ...,True,"{'zero': 9368.0, 'demand': 156.0, 'large': 262..."
1,Final order affects score: retailer,True,"[116.0, 0.0, 592.0]"
2,Final order affects score: wholesaler,True,"[168.0, 0.0, 352.0]"
3,Final order affects score: distributor,True,"[204.0, 0.0, 208.0]"
4,Final order affects score: factory,True,"[224.0, 0.0, 160.0]"
5,"Known beta recovery, including valid zero orders",True,0.65
6,Bullwhip ratio undefined for constant denominator,True,"NaN, not infinity or zero"


Free checks passed. No API calls made by this cell.


## 4. Run the selected stage

This is the only paid cell in API mode. It requests the key in a hidden prompt. Progress shows completed/remaining calls, cached calls, active requests and spending.

A preparation memo reaching its cap is **not a comprehension failure**: the same final-answer allowance follows. Final answers must have a valid nonnegative integer and correct state readbacks. Temporary service errors get bounded retries with pacing/cooldown. A failed final answer may replace the entire four-role group once; failed groups stay recorded and inferential p-values are withheld after replacement. There is no retry-until-good strategy and no automatic model/budget upgrade.

In [8]:
client = None
try:
    if MODE == "api":
        from openai import AsyncOpenAI
        key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("OpenAI API key (hidden): ")
        if not key.strip():
            raise ValueError("No API key entered.")
        client = AsyncOpenAI(api_key=key.strip(), max_retries=0, timeout=CONFIG.timeout_s)
        del key
    run_dir = await run(ROOT, CONFIG, client=client)
    print("Stage complete. Read the report below before proceeding.")
except StudyStopped as exc:
    print("Stopped:", str(exc))
    print("Saved folder:", run_dir)
finally:
    if client is not None:
        await client.close()

Stage complete. Read the report below before proceeding.


## 5. Conclusions and full results

The report leads with conclusions. It saves Word, Markdown, Excel, CSV and plots inside the displayed run folder.

- **H1:** group beta from four identified role estimates; report missing/unreliable fits rather than forcing a value.
- **H2:** total team cost, analyzed independently of beta availability. No p-values from the tiny pilot or synthetic data.
- **Bullwhip (牛鞭效应):** order variance divided by incoming-demand variance at each role, plus order variance divided by external-customer-demand variance for comparison along the chain. Above 1 means amplification relative to that denominator. Also inspect the time series, backlog and inventory. Constant denominators produce missing ratios, not invented zeros.
- The warmup and demand step are retained in the variance window; after the step alone, customer demand is constant. Ratios from this transient experiment are descriptive, not stationary demand estimates.
- Zero orders may follow excess ordering/stock. There is **no advertised order maximum** and no reward for matching a cap. An undisclosed numerical safety check halts absurd-scale orders above 100,000 without clipping them.

[MIT rules and delay dynamics](https://web.mit.edu/jsterman/www/SDG/beergame.html) · [Original bullwhip research](https://pubsonline.informs.org/doi/10.1287/mnsc.43.4.546)

In [9]:
# This cell only reads saved results and writes reports; no simulation/API calls.
# Works even if the kernel was started before the setup cell gained its dependency check.
import importlib.util, subprocess, sys
missing = [package for module, package in
           {"docx": "python-docx>=1,<2", "openpyxl": "openpyxl>=3,<4"}.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
    importlib.invalidate_caches()

if (run_dir / "progress.json").exists():
    display(Markdown(report(run_dir)))
    for name in ("order_trajectories.png", "bullwhip.png"):
        if (run_dir / name).exists():
            display(Image(filename=str(run_dir / name)))
    print("Word report:", run_dir / "REPORT.docx")
    print("Detailed workbook:", run_dir / "DETAILS.xlsx")
    if (run_dir / "calibration_recommendation.json").exists():
        import json
        recommendation = json.loads((run_dir / "calibration_recommendation.json").read_text())
        print("Suggested settings (review before using):", recommendation)
        print("CALIBRATION_RUN =", repr(str(run_dir)))
else:
    print("No saved stage yet.")

# Token-budget Beer Game — conclusions

1. Real API responses; LLM simulation, not human evidence.
2. Run status: complete. Estimated cost/reservations: $0.0540.
3. Saved successful API stages: 192; other saved attempts: 0.
4. Suggested memo caps: 176 / 240 tokens; doubled benchmark 480.
5. Manipulation screen passed: False. This is a provisional feasibility check, not proof of a human time-pressure equivalent.
6. Do not start a paid main experiment yet. Inspect truncation, actual token use and adequate-versus-benchmark agreement.
Calibration uses 24 independent hypothetical states (all four roles), not complete game chains. P25/P95 refer to memo-token use under the 1024-token reference. These percentiles cannot directly represent human response times.

## Calibration summary

```
   screen  mean_tokens  truncated_fraction  mean_order
 adequate     147.7917               0.000      8.3333
benchmark     194.0833               0.000      6.6667
reference     193.2083               0.000      6.2500
    short     140.1250               0.125      5.9167
```

## Calibration state results

```
   screen  item        role  inventory  supply_line  order  plan_tokens  final_tokens  plan_truncated  plan_limit  latency_s
reference     0    retailer          0            0     20          217            39           False        1024     6.1064
reference     1    retailer          0           12      4          203            39           False        1024     3.5020
reference     2    retailer          0           36      0          213            39           False        1024     2.4270
reference     3    retailer         24            0      8          186            39           False        1024     3.6489
reference     4    retailer         24           12      6          189            39           False        1024     3.5273
reference     5    retailer         24           36      8          191            39           False        1024     2.1609
reference     6  wholesaler          0            0     16          149            39           False        1024     2.7733
reference     7  wholesaler          0           12     12          217            39           False        1024     2.6719
reference     8  wholesaler          0           36      0          195            39           False        1024     3.6166
reference     9  wholesaler         24            0      8          201            39           False        1024     2.3751
reference    10  wholesaler         24           12      4          141            39           False        1024     2.0135
reference    11  wholesaler         24           36      4          176            39           False        1024     2.5155
reference    12 distributor          0            0     12          197            39           False        1024     2.1294
reference    13 distributor          0           12     16          225            39           False        1024     3.2515
reference    14 distributor          0           36      0          185            39           False        1024     2.0435
reference    15 distributor         24            0      0          179            39           False        1024     2.8316
reference    16 distributor         24           12      8          176            39           False        1024     2.3025
reference    17 distributor         24           36      0          181            39           False        1024     2.2435
reference    18     factory          0            0     16          207            39           False        1024     2.4025
reference    19     factory          0           12      0          257            39           False        1024     8.1857
reference    20     factory          0           36      0          224            39           False        1024     3.3732
reference    21     factory         24            0      0          137            39           False        1024     2.0790
reference    22     factory         24           12      8          215            39           False        1024     2.8037
reference    23     factory         24           36      0          176            39           False        1024     2.0998
    short     0    retailer          0            0      8          111            39           False         176     1.7493
    short     1    retailer          0           12     16          176            39            True         176     2.3370
    short     2    retailer          0           36      0          157            39           False         176     2.1432
    short     3    retailer         24            0      8          129            39           False         176     1.8628
    short     4    retailer         24           12      0          147            39           False         176     2.0383
    short     5    retailer         24           36      0          114            39           False         176     1.6541
    short     6  wholesaler          0            0     16          149            39           False         176     1.8495
    short     7  wholesaler          0           12     10          128            39           False         176     2.0964
    short     8  wholesaler          0           36      0          176            39           False         176     2.3276
    short     9  wholesaler         24            0      4          113            39           False         176     1.7167
    short    10  wholesaler         24           12      6          163            39           False         176     2.0605
    short    11  wholesaler         24           36     10          113            39           False         176     2.0257
    short    12 distributor          0            0     16          131            39           False         176     1.8217
    short    13 distributor          0           12      8          159            39           False         176     2.2999
    short    14 distributor          0           36      0          153            39           False         176     1.8168
    short    15 distributor         24            0      8          176            39            True         176     2.0379
    short    16 distributor         24           12      8          153            39           False         176     2.0569
    short    17 distributor         24           36      0          122            39           False         176     2.3368
    short    18     factory          0            0     16          118            39           False         176     2.2284
    short    19     factory          0           12      0          176            39            True         176     2.2372
    short    20     factory          0           36      0          128            39           False         176     1.7501
    short    21     factory         24            0      8          142            39           False         176     1.8053
    short    22     factory         24           12      0          132            39           False         176     1.7398
    short    23     factory         24           36      0           97            39           False         176     1.7708
 adequate     0    retailer          0            0     16          162            39           False         240     2.1358
 adequate     1    retailer          0           12     12          148            39           False         240     1.8876
 adequate     2    retailer          0           36     20          123            39           False         240     2.2018
 adequate     3    retailer         24            0      6          125            39           False         240     1.9417
 adequate     4    retailer         24           12      6          173            39           False         240     1.9116
 adequate     5    retailer         24           36      0          137            39           False         240     1.8230
 adequate     6  wholesaler          0            0     12          165            39           False         240     1.8376
 adequate     7  wholesaler          0           12     14          173            39           False         240     2.3376
 adequate     8  wholesaler          0           36      0          138            39           False         240     1.9444
 adequate     9  wholesaler         24            0      8          156            39           False         240     2.0911
 adequate    10  wholesaler         24           12      0          120            39           False         240     1.9844
 adequate    11  wholesaler         24           36      0          131            39           False         240     2.5248
 adequate    12 distributor          0            0     16          217            39           False         240     2.3227
 adequate    13 distributor          0           12     20          139            39           False         240     1.9069
 adequate    14 distributor          0           36     24          195            39           False         240     2.3898
 adequate    15 distributor         24            0      8          128            39           False         240     1.8964
 adequate    16 distributor         24           12      8          157            39           False         240     2.0163
 adequate    17 distributor         24           36      0          163            39           False         240     2.0868
 adequate    18     factory          0            0     16          121            39           False         240     1.9078
 adequate    19     factory          0           12      0          165            39           False         240     2.2988
 adequate    20     factory          0           36      0          145            39           False         240     1.9370
 adequate    21     factory         24            0      8          122            39           False         240     1.8504
 adequate    22     factory         24           12      0          133            39           False         240     1.9703
 adequate    23     factory         24           36      6          111            39           False         240     1.7600
benchmark     0    retailer          0            0     12          193            39           False         480     2.5104
benchmark     1    retailer          0           12     20          164            39           False         480     2.5150
benchmark     2    retailer          0           36     20          173            39           False         480     2.2573
benchmark     3    retailer         24            0      4          189            39           False         480     2.5145
benchmark     4    retailer         24           12      0          160            39           False         480     2.4656
benchmark     5    retailer         24           36      2          213            39           False         480     2.3968
benchmark     6  wholesaler          0            0     12          212            39           False         480     2.3905
benchmark     7  wholesaler          0           12      8          234            39           False         480     2.8176
benchmark     8  wholesaler          0           36      0          212            39           False         480     2.4626
benchmark     9  wholesaler         24            0      8          265            39           False         480     2.4969
benchmark    10  wholesaler         24           12      0          194            39           False         480     2.0356
benchmark    11  wholesaler         24           36      4          183            39           False         480     2.1428
benchmark    12 distributor          0            0     16          159            39           False         480     2.3228
benchmark    13 distributor          0           12     14          227            39           False         480     2.5830
benchmark    14 distributor          0           36     14          272            39           False         480     2.5536
benchmark    15 distributor         24            0      6          175            39           False         480     2.3408
benchmark    16 distributor         24           12      0          175            39           False         480     2.8412
benchmark    17 distributor         24           36      0          169            39           False         480     2.4627
benchmark    18     factory          0            0     16          181            39           False         480     2.3592
benchmark    19     factory          0           12      0          189            39           False         480     2.3470
benchmark    20     factory          0           36      0          173            39           False         480     2.2988
benchmark    21     factory         24            0      0          202            39           False         480     2.0984
benchmark    22     factory         24           12      4          161            39           False         480     2.5308
benchmark    23     factory         24           36      0          183            39           False         480     2.5602
```

## Interpretation limits

This protocol restricts visible preparation tokens plus a common final-response allowance, with reasoning effort none. It does not directly limit hidden cognitive effort, wall-clock time or human thinking time. Prompts cannot reliably count remaining tokens; the API hard cap is the enforced constraint. A truncated memo is accepted as a deadline outcome, then a common 192-token stage solicits the order. The final stage can still involve implicit computation, and unused token allowance is not effort used.
Technical retries retain the same prompt and cap. Failed final responses can replace a whole group only within the configured limit; all failed attempts remain recorded. Replacement selection withholds inferential p-values. No order is clipped or replaced by a fabricated answer. The separate emergency scale check stops the run rather than making a boundary order.

Sources: [provided reflection](../../source/ThoughtsReflection_Paragraph.docx); [MIT Beer Game](https://web.mit.edu/jsterman/www/SDG/beergame.html); [Lee, Padmanabhan & Whang](https://pubsonline.informs.org/doi/10.1287/mnsc.43.4.546); [OpenAI model documentation](https://developers.openai.com/api/docs/models/gpt-5.4-nano).


Word report: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6778221cada613b6/REPORT.docx
Detailed workbook: /Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6778221cada613b6/DETAILS.xlsx
Suggested settings (review before using): {'short_tokens': 176, 'adequate_tokens': 240, 'benchmark_tokens': 480, 'median_absolute_adequate_benchmark_order_difference': 2.0, 'manipulation_screen_passed': False, 'mode': 'api', 'note': 'Pilot thresholds, not validated psychometric cutoffs. No automatic main study.'}
CALIBRATION_RUN = '/Users/gracezhang/Desktop/Y3S1/Project/beer_game_experiment/data/runs/calibration_api_6778221cada613b6'


## 6. What to do next

1. Run this entire notebook in **offline calibration** mode once.
2. Run **API calibration**; review whether the cap binds and the generous/doubled decisions are reasonably stable. A failed screen means the token contrast is not yet established; do not call it a successful time-pressure manipulation.
3. For a passed screen, copy its two budgets and folder into the settings, change `STAGE="game"`, and run **one group per condition** first. Keep `REPLICATE_ID` fixed to resume.
4. Assess beta identification, total costs, actual token use, failure rates and bullwhip patterns. Plan group-level replication/power before the proposed **10 groups per condition**. The previous $3 budget is not assumed sufficient for 80 agents.

No prompt, token budget or model guarantees realistic human behavior, bullwhip amplification or significant H1/H2. The study tests those outcomes.

The updated reflection also proposes explanations/questionnaires. Set `COLLECT_DEBRIEFS=True` **before a new game run** to save one concise retrospective self-report per agent. This is optional, adds API cost, and is separate from decision data. Model self-reports do not establish the actual mechanism behind decisions.